# Python Read Files

> 📘 **Python Mastery** · Module 06 — File Handling · Lesson 1/4

Variables are forgetful — every value vanishes the moment your script ends. **Files** give your data a memory. This lesson shows you how to open a file and read it back safely, whether it holds three lines or three gigabytes.

## 🎯 Learning Objectives

By the end of this lesson you will be able to:

- Explain what `open()` does and pick the right **mode**: `r`, `w`, `a`, `x`, combined with `t` or `b`.
- Read a file four ways — `.read()`, `.read(n)`, `.readline()`, `.readlines()` — plus the memory-friendly `for` loop.
- Use `with open(...) as f:` and explain why it beats calling `.close()` yourself.
- Handle `FileNotFoundError` gracefully and check existence with `pathlib.Path.exists()`.
- Tell absolute from relative paths, and use `pathlib` for portable path handling.
- Justify always passing `encoding="utf-8"` when working with text files.

## 1. What `open()` Actually Does

A file on disk is just a long sequence of bytes. `open()` asks the operating system for a **file object** — think of it as a borrowing slip that lets your program walk through those bytes. Importantly, opening a file does *not* dump the whole thing into memory; the file object pulls in pieces only as you request them.

**Syntax:**

```python
file_object = open("path/to/file.txt", mode="r", encoding="utf-8")
#   mode     -> what you intend to do: read, write, append, create
#   encoding -> how characters map to bytes (always "utf-8" for text)
```

In [1]:
from pathlib import Path

# Every lesson in this module works inside a sample_data/ folder.
Path("sample_data").mkdir(exist_ok=True)

# Create a small file first, so this notebook is fully self-contained.
content = """Sarah's Study Log
Day 1: Learned about variables and numbers.
Day 2: Wrote my first if-statement.
Day 3: Loops finally make sense!
"""
Path("sample_data", "sample.txt").write_text(content, encoding="utf-8")
print("sample_data/sample.txt created:", Path("sample_data", "sample.txt").exists())

sample_data/sample.txt created: True


## 2. File Modes: the Second Argument

The mode string tells `open()` your intention *before* it touches the file. It combines one **action letter** with one optional **type letter**.

| Mode | Name | Behaviour |
|------|------|-----------|
| `"r"` | read | Read an **existing** file. Missing file → `FileNotFoundError`. *(default)* |
| `"w"` | write | Create a new file — or **wipe** an existing one. (Lesson 2) |
| `"a"` | append | Keep existing content, add new data at the end. Creates if missing. (Lesson 3) |
| `"x"` | exclusive create | Create, but **refuse** if the file already exists (`FileExistsError`). (Lesson 3) |
| `"t"` | text mode | You work with `str`. *(default)* |
| `"b"` | binary mode | You work with `bytes` — images, audio, saved ML models. |

Modes combine freely: `"rb"` = read binary, `"wt"` = write text, `"a+"` = append **and** read back. Lessons 2 and 3 cover the writing side.

In [2]:
# Text mode hands you str; binary mode hands you bytes.
with open("sample_data/sample.txt", "r", encoding="utf-8") as f:
    print("text mode   ->", repr(f.readline()))

with open("sample_data/sample.txt", "rb") as f:
    print("binary mode ->", repr(f.readline()))

text mode   -> "Sarah's Study Log\n"
binary mode -> b"Sarah's Study Log\r\n"


## 3. Reading Everything: `.read()` and `.read(n)`

`.read()` with no argument returns the **entire file** as one string. `.read(n)` returns at most the next `n` **characters** and remembers where it stopped — handy for chunked processing of big files.

**Syntax:**

```python
whole_file = f.read()      # everything, as one big string
chunk      = f.read(100)   # the next 100 characters only
```

**Example:**

In [3]:
with open("sample_data/sample.txt", encoding="utf-8") as f:
    text = f.read()          # slurp the whole file
print(text)
print("Characters in file:", len(text))

Sarah's Study Log
Day 1: Learned about variables and numbers.
Day 2: Wrote my first if-statement.
Day 3: Loops finally make sense!

Characters in file: 131


In [4]:
with open("sample_data/sample.txt", encoding="utf-8") as f:
    head = f.read(10)        # first 10 characters...
    tail = f.read()          # ...then whatever remains
print("head:", repr(head))
print("tail starts with:", repr(tail[:22]))

head: "Sarah's St"
tail starts with: 'udy Log\nDay 1: Learned'


## 4. Reading Line by Line: `readline()`, `readlines()`, and `for`

Text files usually make sense one line at a time. Python gives you three tools, from "one at a time by hand" to "all at once":

- `f.readline()` → the **next single line**, including its `\n`. Returns an empty string `""` when the file is exhausted.
- `f.readlines()` → a **list** of all lines at once (fine for small files, hungry for big ones).
- `for line in f:` → yields **one line per iteration**, loading as it goes. This is the idiomatic, memory-efficient choice.

**Syntax:**

```python
first_line = f.readline()
all_lines  = f.readlines()          # list of strings

for line in f:                      # best for large files
    process(line)
```

**Example:**

In [5]:
with open("sample_data/sample.txt", encoding="utf-8") as f:
    line1 = f.readline()
    line2 = f.readline()
    line4 = f.readline()            # skip line 3 on purpose
print("raw line 1:", repr(line1))   # note the \n riding along
print("tidy line 2:", line2.strip())
print("line 4     :", repr(line4))

raw line 1: "Sarah's Study Log\n"
tidy line 2: Day 1: Learned about variables and numbers.
line 4     : 'Day 2: Wrote my first if-statement.\n'


In [6]:
with open("sample_data/sample.txt", encoding="utf-8") as f:
    lines = f.readlines()           # a list, one string per line

print("Read", len(lines), "lines into a list:")
for number, line in enumerate(lines, start=1):
    print(number, "->", line.strip())   # strip() trims the trailing \n

Read 4 lines into a list:
1 -> Sarah's Study Log
2 -> Day 1: Learned about variables and numbers.
3 -> Day 2: Wrote my first if-statement.
4 -> Day 3: Loops finally make sense!


In [7]:
# The idiomatic pattern: loop straight over the file object.
with open("sample_data/sample.txt", encoding="utf-8") as f:
    for number, line in enumerate(f, start=1):
        print(f"{number}: {line.strip()}")

1: Sarah's Study Log
2: Day 1: Learned about variables and numbers.
3: Day 2: Wrote my first if-statement.
4: Day 3: Loops finally make sense!


> 🔍 **Under the Hood:** the file object secretly keeps an integer called the **cursor** (position) — how many characters you have consumed so far. `.read(n)`, `.readline()` and the `for` loop all advance it; `.read()` from the start consumes it all. Because the `for` loop only materialises *one* line at a time, streaming a 50 GB log file costs kilobytes of RAM, whereas `.read()` or `.readlines()` would try to fit all 50 GB in memory and crash. That is why production code streams with `for line in f`.

In [8]:
# Same file, two memory profiles.
with open("sample_data/sample.txt", encoding="utf-8") as f:
    as_list = f.readlines()             # EVERY line held in memory at once
print("readlines():", type(as_list).__name__, "holding", len(as_list), "strings")

with open("sample_data/sample.txt", encoding="utf-8") as f:
    biggest = 0
    for line in f:                      # only ONE line alive per iteration
        biggest = max(biggest, len(line))
print("for loop   : peak extra memory = one line,", biggest, "chars max")

readlines(): list holding 4 strings
for loop   : peak extra memory = one line, 44 chars max


## 5. Closing Files Properly: `.close()` and `with`

An open file holds an operating-system resource. On Windows an unclosed file stays **locked** against other programs; on any OS, buffered data can be lost if the program exits badly. You *can* close files by hand — but if an exception fires before `.close()`, the file leaks. The `with` statement guarantees closure no matter what.

**Syntax:**

```python
# Manual (works, but fragile)
f = open("data.txt", encoding="utf-8")
try:
    text = f.read()
finally:
    f.close()

# Idiomatic (auto-closes, even on errors)
with open("data.txt", encoding="utf-8") as f:
    text = f.read()
```

**Example:**

In [9]:
# Manual close: you are responsible for remembering it.
f = open("sample_data/sample.txt", encoding="utf-8")
text = f.read()
f.close()
print("After manual .close(), closed =", f.closed)
# If an exception had struck between open() and close(),
# this file would have leaked. with fixes that next.

After manual .close(), closed = True


In [10]:
# The with statement: Python closes the file FOR you.
with open("sample_data/sample.txt", encoding="utf-8") as f:
    text = f.read()
    print("Inside the block , closed =", f.closed)

print("After the  block , closed =", f.closed)   # <- automatic!

Inside the block , closed = False
After the  block , closed = True


Why does `with` know how to clean up?

> 🔍 **Under the Hood:** `with` uses Python's **context-manager protocol**. Entering the block calls the object's `__enter__()` method (whose return value is bound to `f`); leaving the block — through the bottom *or* through an exception — calls `__exit__()`, and the file's `__exit__()` closes the stream. Any class defining these two dunder methods works with `with`: database connections, network sockets, locks, even your own classes (you will build one in the OOP module). `with` is simply a promise: *"this resource gets released on every path."*

## 6. Missing Files: `FileNotFoundError` and `.exists()`

Reading a file that is not there raises `FileNotFoundError`. Production code expects this: catch it and recover (use a default, warn the user, create the file) rather than letting the program crash. If you merely want to *check* first, `pathlib.Path.exists()` answers without touching the file — though the check and the open are two separate moments, so the try/except version is still the race-proof choice.

**Syntax:**

```python
try:
    with open(path, encoding="utf-8") as f:
        ...
except FileNotFoundError:
    ...   # recover politely

Path(path).exists()   # True/False, no exception
```

**Example:**

In [11]:
try:
    with open("sample_data/no_such_file.txt", encoding="utf-8") as f:
        print(f.read())
except FileNotFoundError:
    print("Caught it! That file really does not exist.")

Caught it! That file really does not exist.


In [12]:
from pathlib import Path

guess = Path("sample_data", "missing.txt")
print(guess.exists())            # False -> no need to even try opening

real = Path("sample_data", "sample.txt")
if real.exists():
    print(real.name, "is ready to be read.")

False
sample.txt is ready to be read.


## 7. Paths: Relative, Absolute, and `pathlib`

A **relative path** (`"sample_data/sample.txt"`) is resolved from the **current working directory** — the folder your program runs from, not necessarily where the script lives. An **absolute path** starts from the drive root (`H:\...` or `/home/...`) and works from anywhere, but ties your code to one machine. Modern Python prefers `pathlib.Path`, which joins paths with `/`, works identically on Windows/Mac/Linux, and offers shortcuts like `read_text()` that open-and-close the file in one line.

**Syntax:**

```python
from pathlib import Path

p = Path("sample_data") / "sample.txt"   # join paths portably
p.resolve()                              # relative -> absolute
p.read_text(encoding="utf-8")            # open + read + close, one line
p.write_text("hi", encoding="utf-8")     # same trick for writing
```

**Example:**

In [13]:
from pathlib import Path

relative = Path("sample_data", "sample.txt")
absolute = relative.resolve()          # full path from the drive root

print("Relative :", relative)
print("Absolute :", absolute)

# The modern shortcut: open, read, close — one expression.
first_line = relative.read_text(encoding="utf-8").splitlines()[0]
print("First line:", first_line)

Relative : sample_data\sample.txt
Absolute : H:\Python-Mastery\06_File_Handling\01_Read_File\sample_data\sample.txt
First line: Sarah's Study Log


## 8. Encoding: Why `utf-8` Matters

Computers store numbers, not letters. An **encoding** is the agreement about which number means which character. UTF-8 covers every language and emoji, and it is the web standard — but `open()` without an explicit `encoding=` falls back to whatever the local machine prefers (on Windows often `cp1252`, which knows far fewer characters). The result: a file that opens fine on your laptop raises `UnicodeDecodeError` on a colleague's machine, or worse, loads as *mojibake* (`Ã©` instead of `é`). One habit fixes it forever: always say `encoding="utf-8"`.

**Syntax:**

```python
open(path, encoding="utf-8")            # reading
Path(path).write_text(text, encoding="utf-8")   # writing
```

**Example:**

In [14]:
from pathlib import Path

note = "Cafe bill: 250 taka, paid with a smile ✓"
path = Path("sample_data", "unicode_note.txt")

# 1) Naive: let the machine guess the encoding.
try:
    path.write_text(note)               # no encoding given!
    print("Machine-default encoding handled it on THIS computer.")
except UnicodeEncodeError:
    print("Machine-default encoding FAILED for these characters!")

# 2) Professional: spell it out, works on every OS, every machine.
path.write_text(note, encoding="utf-8")
print("UTF-8 round-trip:", path.read_text(encoding="utf-8"))

Machine-default encoding FAILED for these characters!
UTF-8 round-trip: Cafe bill: 250 taka, paid with a smile ✓


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| Opening without `with` | File stays locked (Windows) and may leak if an error fires mid-way | Always `with open(...) as f:` |
| `.read()` on a huge file | Tries to fit gigabytes into RAM → `MemoryError` | Stream with `for line in f:` |
| Comparing raw lines to expected text | Hidden `\n` makes `"Day 1"` != `"Day 1\n"` | `line.strip()` or `line.rstrip("\n")` |
| Hard-coding absolute paths like `H:\\mine\\data.txt` | Breaks on every other machine | Relative paths built with `pathlib`, verified with `.exists()` |
| Skipping `encoding="utf-8"` | Crashes or garbled text on machines with other defaults | Pass it explicitly, every time |
| Catching `FileNotFoundError` with a bare `except:` | Hides unrelated bugs (typos, permissions) | `except FileNotFoundError:` specifically |

## 💡 Best Practices & Pro Tips

- Make `with open(..., encoding="utf-8")` your default reflex — the two times you should not use it are binary mode and exotic streaming APIs.
- Prefer `pathlib.Path` objects over string paths: they join with `/`, compare cleanly, and pair with `.exists()`, `.glob()` and `.read_text()`.
- Know your file size before choosing a strategy: `.read()` for config files and small datasets, streaming loops for logs and exports.
- Build paths from named pieces (`Path("sample_data") / "students.csv"`), never by gluing strings with `+`.
- **AI-engineering relevance:** almost every pipeline starts by *reading*: training corpora streamed line-by-line, inference logs tailed, model configs and label maps loaded at startup. Streaming readers and explicit UTF-8 are exactly what keeps those jobs alive on a cluster — and `FileNotFoundError` handling is what turns "crashed at 2 a.m." into "fell back to yesterday's checkpoint".

## 📌 Summary

| Tool | What it does | Example |
|------|--------------|---------|
| `open(path, mode, encoding)` | Returns a file object | `open("a.txt", "r", encoding="utf-8")` |
| `.read()` | Whole file as one string | `text = f.read()` |
| `.read(n)` | Next `n` characters | `chunk = f.read(1024)` |
| `.readline()` | One line (with `\n`) | `line = f.readline()` |
| `.readlines()` | List of all lines | `lines = f.readlines()` |
| `for line in f` | Memory-efficient streaming | `for line in f: ...` |
| `with ... as f` | Auto-closes, even on errors | `with open(p) as f:` |
| `f.closed` | `True` once the file is closed | `assert f.closed` |
| `Path.read_text()` | One-line read via pathlib | `Path(p).read_text(encoding="utf-8")` |
| `Path.exists()` | Existence check, no exception | `Path(p).exists()` |

Key takeaways:

- `open()` returns a *handle*, not the data — you choose how much to pull with `.read*()` or a loop.
- `"r"` + `"t"` are the defaults; `"b"` switches to bytes for non-text content.
- `with` is not style, it is safety: it closes the file on every possible exit path.
- Stream big files line-by-line, and never let an `encoding=` go unspecified.

## 🔗 Next Lesson

Reading is half of the conversation. Next, in [`../02_Write_File/notes.ipynb`](../02_Write_File/notes.ipynb), you learn how to **write** files — including the one mode that silently destroys data if you misuse it.